**1. Monthly Revenue Trend - Merge Sales, Products, Exchange Rates. Calculate total revenue USD by month (2024)**

In [0]:
SELECT 
    d.year,
    d.month,
    d.month_name,
    SUM(f.revenue_usd) AS total_revenue
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_date d
    ON f.order_date = d.order_date
WHERE d.year = 2020
GROUP BY d.year, d.month, d.month_name
ORDER BY d.month;

**2. Peak Month Analysis - From Q1, identify top 3 revenue months. Calculate % of annual total**


In [0]:
WITH monthly_revenue AS (
    SELECT 
        d.month,
        SUM(f.revenue_usd) AS revenue
    FROM 03_prod_gold.models.fact_sales f
    JOIN 03_prod_gold.models.dim_date d
        ON f.order_date = d.order_date
    WHERE d.year = 2020
    GROUP BY d.month
),
total_revenue AS (
    SELECT SUM(revenue) AS total FROM monthly_revenue
)

SELECT 
    m.month,
    m.revenue,
    ROUND((m.revenue / t.total) * 100, 2) AS percent_contribution
FROM monthly_revenue m
CROSS JOIN total_revenue t
ORDER BY m.revenue DESC
LIMIT 3;

**3. Holiday Drivers - For Q2 peak months, show top 3 product categories driving revenue**


In [0]:
WITH top_months AS (
    SELECT 
        d.month
    FROM 03_prod_gold.models.fact_sales f
    JOIN 03_prod_gold.models.dim_date d
        ON f.order_date = d.order_date
    WHERE d.year = 2020
    GROUP BY d.month
    ORDER BY SUM(f.revenue_usd) DESC
    LIMIT 3
)

SELECT 
    p.category,
    SUM(f.revenue_usd) AS revenue
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_products p
    ON f.product_key = p.product_key
JOIN 03_prod_gold.models.dim_date d
    ON f.order_date = d.order_date
WHERE d.month IN (SELECT month FROM top_months)
GROUP BY p.category
ORDER BY revenue DESC
LIMIT 3;

**4. Delivery Performance - Calculate overall avg delivery time across all orders**

In [0]:
SELECT 
    AVG(delivery_days) AS avg_delivery_days
FROM 03_prod_gold.models.fact_sales
WHERE store_key=0;

**5. Country Delivery Issue - Avg delivery time by store country. Show slowest 5 countries**

In [0]:
SELECT 
    s.country,
    AVG(f.delivery_days) AS avg_delivery_days
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_stores s
    ON f.store_key = s.store_key
WHERE f.store_key=0
GROUP BY s.country
ORDER BY avg_delivery_days DESC
LIMIT 5;

**6. Channel Performance - AOV (revenue/orders) by Online vs In-Store across continents (Online = StoreKey.isna())**

In [0]:
SELECT 
    c.continent,
    CASE 
        WHEN f.store_key = 0 THEN 'Online'
        ELSE 'In-Store'
    END AS channel,
    SUM(f.revenue_usd) / COUNT(DISTINCT f.order_number) AS aov
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY c.continent, channel
ORDER BY c.continent, aov DESC;

**7. Volume Leaders - Top 5 product categories by total units sold**

In [0]:
SELECT 
    p.category,
    SUM(f.quantity) AS total_units
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_units DESC
LIMIT 5;

**8. Revenue Leaders - Top 5 product categories by total revenue USD**

In [0]:
SELECT 
    p.category,
    SUM(f.revenue_usd) AS total_revenue
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_products p
    ON f.product_key = p.product_key
GROUP BY p.category
ORDER BY total_revenue DESC
LIMIT 5;

**9. Customer Profile - Customer count and spending by Continent × Gender**

In [0]:
SELECT 
    c.continent,
    c.gender,
    COUNT(DISTINCT f.customer_key) AS customer_count,
    SUM(f.revenue_usd) AS total_spend
FROM 03_prod_gold.models.fact_sales f
JOIN 03_prod_gold.models.dim_customers c
    ON f.customer_key = c.customer_key
GROUP BY c.continent, c.gender
ORDER BY c.continent;

**10. Customer Loyalty - Repeat customer rate (% with 2+ orders) by continent**

In [0]:
WITH customer_orders AS (
    SELECT 
        f.customer_key,
        c.continent,
        COUNT(DISTINCT f.order_number) AS order_count
    FROM 03_prod_gold.models.fact_sales f
    JOIN 03_prod_gold.models.dim_customers c
        ON f.customer_key = c.customer_key
    GROUP BY f.customer_key, c.continent
)

SELECT 
    continent,
    ROUND(
        SUM(CASE WHEN order_count >= 2 THEN 1 ELSE 0 END) * 100.0
        / COUNT(*), 2
    ) AS repeat_customer_percentage
FROM customer_orders
GROUP BY continent;